In [ ]:
import os

from modules.context import get_concepts

from pathlib import Path
os.chdir(Path.cwd().parent)
import time
import copy
import json
import random
import warnings
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.core.magic import register_cell_magic

from data.dataloader import load_dataset
from utils.utils import set_random_seed, Usage
from utils.llm import load_model
from utils.metrics import get_metrics_fn
from utils.multiprocess import multiprocess_call
from modules.post_process import get_label_tokenizer, get_logit, prepare_labels, match_drug_fuzz
from TherapyReason.significance import get_significance
from TherapyReason.therapy import get_dl_candidates, rank_dl_candidates
from TherapyReason.reasoning import candidate2string, template_solution_dl, solution2string_with_overview, \
    post_process_solution_wrapper

set_random_seed(42)
dataset = 'mimic3'
task = 'drugrec'
label_key = 'drugs'
datasets, full_set = load_dataset(dataset, task)
test_set = datasets['test']
label_tokenizer = get_label_tokenizer(full_set, label_key)
cond_tokenizer = get_label_tokenizer(full_set, 'conditions')
proc_tokenizer = get_label_tokenizer(full_set, 'procedures')
drug_tokenizer = label_tokenizer
significance = get_significance(dataset, task)
cache_dir = f"./TherapyReason/cache/{dataset}-{task}"
os.makedirs(cache_dir, exist_ok=True)
drug_similarity = json.loads(Path("TherapyReason/cache/llm_drug_pairwise.json").read_text())
print(len(test_set))

warnings.filterwarnings('ignore')


@register_cell_magic
def skip(line, cell): return None


llm = load_model({"model": "DeepSeek-V3", "server": "ant3", "do_sample": False, "temperature": 0.0, "max_attempts": 10})

In [ ]:
# Candidates
thres = 0.4
sample_dl_candidates = get_dl_candidates(thres, os.path.join(cache_dir, "sensitivity"), dataset, task, full_set, test_set)
sample_candidate_ranks = rank_dl_candidates(os.path.join(cache_dir, "sensitivity"), dataset, task, full_set, test_set)

In [ ]:
sample_solutions = []

for i in range(len(test_set)):
    patient = test_set[i]
    dl_candidates = sample_dl_candidates[i]
    conditions, procedures, _ = get_concepts(patient, task, with_history=True, flatten=False)

    solutions_visits = []
    for vid in range(len(conditions)):
        solution = {
            "header": None,
            "conditions": conditions[vid],
            "procedures": procedures[vid],
            "drugs": [],
            "dl_drugs": dl_candidates
        }
        solutions = [solution]
        solutions_visits.append(solutions)

    sample_solutions.append({
        "solutions": solutions_visits,
        label_key: patient[label_key],
        "isolates": []
    })

In [ ]:
from TherapyReason.overview import post_process_solution_overview

template_solution_overview = """\
# Task: Patient Overview
Go over the key information in the patient's context. Summarize the patient's context in one paragraph briefly.

## Patient's Context
{solution_string}

## Output Format
Patient Overview: [Your output]
"""

def solution2string(solution):
    solution_name, solution_path = "", ""
    if solution['conditions']:
        solution_name += "### Conditions\n" + "".join(x + "\n" for x in solution['conditions'])
    if solution['procedures']:
        solution_name += "### Procedures\n" + "".join(x + "\n" for x in solution['procedures'])
    return solution_name

def process_solution_overview_wrap(cache_dir, llm):
    def process_solution_overview(key, item, **kwargs):
        cache_file = Path(os.path.join(cache_dir, f"{key}.json"))
        os.makedirs(cache_file.parent, exist_ok=True)
        if cache_file.exists():
            cached = json.loads(cache_file.read_text(encoding="utf-8"))
        else:
            time.sleep(random.randint(1, 10))
            solution_obj = item

            cached = []
            for visit_id, visit in enumerate(solution_obj['solutions']):
                cached_visit = []
                for solution_id, solution in enumerate(visit):
                    assert solution_id == 0
                    query = template_solution_overview.format(solution_string=solution2string(solution))
                    _, overview, _, usage = llm.understand(query, post_process=post_process_solution_overview, **kwargs)
                    cached_visit.append({"query": query, "overview": overview, "usage": usage.dict()})
                cached.append(cached_visit)
            cache_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")

        return key, cached

    return process_solution_overview

In [ ]:
# solution overview
process_solution_overview = process_solution_overview_wrap(os.path.join(cache_dir, "ablation", f"solution_thres{int(thres * 10)}_nokg_overview", llm.genre), llm)
solution_overview_output = multiprocess_call(process_solution_overview, sample_solutions, n_workers=8, verbose=False)

## solution reasoning

In [ ]:
# 初始化solution reasoning的cache文件（写空文件）
n_shot = 2
chunk_size = 20

for key, item in enumerate(list(zip(sample_solutions, solution_overview_output))):
    solution_obj, solution_overview = item

    cache_dl_file = Path(os.path.join(cache_dir, "ablation", f"solution_thres{int(thres * 10)}_nokg_dl", llm.genre, f"{key}.json"))
    os.makedirs(os.path.dirname(cache_dl_file), exist_ok=True)
    if not cache_dl_file.exists():
        cached = []
        for visit_id, visit in enumerate(solution_obj['solutions']):
            _cached = []
            for solution_id, solution in enumerate(visit):
                assert solution_id == 0
                n_chunks = int(np.ceil(len(solution["dl_drugs"]) / chunk_size))
                _cached.append({"dl_drugs": solution["dl_drugs"], "dl_query": [None] * n_chunks, "dl_response": [None] * n_chunks,
                    "dl_prediction": {}, "dl_usage": Usage().dict()})
            cached.append(_cached)
        cache_dl_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")
        print(f"Init {cache_dl_file}")

In [ ]:
from TherapyReason.reasoning import extract_json_blocks
def post_process_solution_wrapper(drug_tokenizer, n_candidates, candidate_dict):
    def post_process_solution(prob, response, output):
        scoring = {}
        response = response.replace("*", "")
        for drug, item in extract_json_blocks(response).items():
            drug = match_drug_fuzz(drug, drug_tokenizer.vocabulary.token2idx.keys())
            item = {"reason": item['reason'], "predict": int(item["predict"])}
            scoring[drug] = item

        for drug in candidate_dict:
            if drug not in scoring and len(candidate_dict[drug]) == 0:
                scoring[drug] = {"reason": "", "predict": 0}

        assert len(scoring) == n_candidates
        return prob, (response, scoring), output

    return post_process_solution

In [ ]:
# 对DL进行reasoning
template_solution_system = """\
# Task: Drug Recommendation
Score drugs based on their suitability in the patient's healthcare based on conditions and procedures.
Scores: 0 (Unsuitable), 1 (Poor fit), 2 (Possible), 3 (Recommended), 4 (Optimal)

## Key Considerations
Conditions:
- Target: Specific diseases/indications the drug treats (e.g. diabetes, hypertension, infection type)
- Severity: Stage/severity of disease (e.g. mild vs. end-stage)
- Comorbidities: Conditions that may contraindicate drugs (e.g. renal failure, liver disease)
- Supportive: Drugs not directly treat but can support the healthcare (e.g. alleviate symptoms, supportive drugs)
Procedures:
- Recent surgeries: Procedures affecting drug metabolism (e.g. GI surgeries impacting absorption)
- Invasive interventions: Lines/ports available for administration (e.g. IV-only drugs if no central line)
- Monitoring capacity: Whether required monitoring is feasible (e.g. frequent lab tests for anticoagulants)
"""

def candidate2string(candidates, solution):
    candidate_string_dict = {node: [] for node in candidates}
    return dict(candidate_string_dict)

def process_solutions_dl(key, item, **kwargs):
    solution_obj, solution_overview = item
    visits = solution_obj['solutions']

    cache_file = Path(os.path.join(cache_dir, "ablation", f"solution_thres{int(thres * 10)}_nokg_dl", llm.genre, f"{key}.json"))
    cached = json.loads(cache_file.read_text(encoding="utf-8"))
    for visit_id, visit in enumerate(visits):
        for solution_id, solution in enumerate(visit):
            assert solution_id == 0
            overview = solution_overview[visit_id][solution_id]['overview']

            n_chunks = int(np.ceil(len(solution['dl_drugs']) / chunk_size).item())
            for i in range(n_chunks):
                if cached[visit_id][solution_id]['dl_response'][i] is not None: continue

                time.sleep(random.randint(0, 4))
                print(f"Patient: {key}\tVisit: {visit_id + 1}/{len(visits)}\tSolution: {solution_id + 1}/{len(visit)}\tDL Chunk: {i}/{n_chunks}")
                candidate_string_dict = candidate2string(solution["dl_drugs"][i * chunk_size: (i + 1) * chunk_size], solution)

                post_process_solution = post_process_solution_wrapper(drug_tokenizer, candidates=list(candidate_string_dict))
                query = template_solution_dl.format(solution_string=solution2string_with_overview(solution, overview, overview_only=True),
                                                    candidate_dict=json.dumps(candidate_string_dict, indent=2), candidates="\t".join(candidate_string_dict))
                _, (response, prediction), _, usage = llm.understand(query, system=template_solution_system, post_process=post_process_solution, **kwargs)
                cached[visit_id][solution_id]['dl_query'][i] = query
                cached[visit_id][solution_id]['dl_response'][i] = response
                cached[visit_id][solution_id]['dl_prediction'].update(prediction)
                cached[visit_id][solution_id]['dl_usage'] = (usage + cached[visit_id][solution_id]['dl_usage']).dict()
                cache_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")

    return key, cached


sample_dl_reasoning = multiprocess_call(process_solutions_dl, list(zip(sample_solutions, solution_overview_output)), n_workers=8, verbose=False)

# eval

In [ ]:
# 评估DL的结果
def get_dl_metric(offset, threshold, dl_reasoning, agg='mean'):
    y_true_all, y_prob_all = [], []
    for i in range(len(test_set)):
        patient_prediction = defaultdict(list)
        for visit_id, visit in enumerate(dl_reasoning[i]):
            for solution_id in range(len(visit)):
                for drug, value in dl_reasoning[i][visit_id][solution_id]['dl_prediction'].items():
                    assert drug is not None, i
                    patient_prediction[drug].append(value['predict'] + offset)

        patient_prediction = {drug: np.mean(scores) if agg == 'mean' else np.max(scores) for drug, scores in patient_prediction.items()}
        patient_prediction = [drug for drug, score in patient_prediction.items() if score >= threshold]
        y_true_all.append(prepare_labels(task, [test_set[i][label_key]], label_tokenizer)[0].numpy())
        y_prob_all.append(get_logit(task, list(patient_prediction), label_tokenizer))

    metrics = get_metrics_fn(task)(np.stack(y_true_all), np.array(y_prob_all))
    print(offset, threshold, agg, metrics["precision_samples"], metrics["recall_samples"], metrics["f1_samples"], metrics["jaccard_samples"], metrics['hamming_loss'], metrics['rank'])

for agg in ['max', 'mean']:
    for offset in range(2, 4):
        for threshold in range(offset + 1, min(offset + 5, 5)):
            get_dl_metric(offset, threshold, sample_dl_reasoning, agg=agg)